<a href="https://colab.research.google.com/github/AngelinaBhowmick1/skills-review-pull-requests/blob/main/assignmentcogs117gpt_basics_tutorial_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# A gentle introduction to how GPT works

The following notebook is a short tutorial on some of the basics of language transformers. You don't need to have much or any background in neural networks. It's helpful if you have some beginner knowledge of Python (so that you can make sense of the code more easily), but it's not necessary to work through the notebook.

Focus on working through the text instructions, answering the questions, and trying out the code chunks where you can play around with the models. The goal is not for you to understand all of the code, but it's there for those of you who are curious and excited to dig deeper.

**Goal:** Your goal is to understand three big ideas:
1. **Prediction** — GPT predicts the next token
2. **Self-attention** — GPT decides which earlier words matter
3. **Representations** — words get turned into vectors whose meaning depends on context

Before working through this notebook, I recommend you read the following blog post, a gentle introduction to transformer-style LLMs:
https://arstechnica.com/science/2023/07/a-jargon-free-explanation-of-how-ai-large-language-models-work/

---

## Before you start

First, **make your own copy of the notebook and put your name in the text chunk below**. To submit the assignment, you will share the link with us. Make sure that the link is accessible, by either making the link accessible to anyone or (if you want to keep it private) sharing the document with me, and your TAs/PLAs:
- mzettersten@ucsd.edu
- vhennessy@ucsd.edu
- apc003@ucsd.edu
- mewolfe@ucsd.edu
- zeh012@ucsd.edu

---

## How to use this notebook

Please move slowly. Run **one cell at a time**.

Whenever you see a section called **Try it yourself**, stop and type your own example (in the dedicated code chunk).

Whenever you see **Questions**, write a short answer in a comment or markdown cell before moving on.

There are also **Bonus Questions**. These are optional, but answering them can net you up to one bonus point on the assignment.

This notebook will be graded based on completion. Attempting all **Try it yourself** and **Questions** sections will net full points (just show us that you tried them out, even if you're not sure about the answer).

### Why we are doing this
The goal here is not for you to become an expert on LLMs or to understand what every line of the code within some of the helper functions is doing, so don't worry about some of the pieces potentially being confusing. The goal is to try to interact a little bit more with a *real* GPT model, to build some of your intuitions about the basics.

*Side note: I'm giving you the full code chunks so that those of you who are more experienced programmers can also see a little more under the hood and get some sample "clay" to play with.*

### Just a quick note

There are no real guardrails about what the model we are using might generate (that's one of the downsides of exploring an open-source model like this). This means it's entirely possible that it might generate text that is upsetting as you are exploring. If you have any concerns, please just let one of our instructors know.




## Please enter your name below

**Name:**


## 0. Setup

This notebook uses a **small GPT-2 style model** from Hugging Face so that we can inspect what is happening under the hood.

In Google Colab, just run the next cell.

**These next two lines run the longest, because we're loading the model (should be <1 minute).**


In [ ]:

# run this line once in Google Colab:
!pip -q install transformers torch matplotlib pandas

import torch
import math
import pandas as pd
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


In [ ]:

# Load a small GPT-style model
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    output_attentions=True,
    output_hidden_states=True
).to(device)

model.eval()

print("Model loaded:", model_name)
print("Vocabulary size:", model.config.vocab_size)
print("Number of layers:", model.config.n_layer)
print("Number of attention heads:", model.config.n_head)
print("Hidden size:", model.config.n_embd)



## 1. The core idea: GPT predicts the next token

The basic training task of a GPT model is extremely simple:

> Given the tokens so far, predict the next token.

That sounds small, but it turns out that doing this very well forces the model to learn a lot about syntax, meaning, and world knowledge.

### Important note
GPT predicts the next **token**, not always the next full word.  
A token might be:
- a whole word
- part of a word
- punctuation
- even a leading space

Let's inspect tokenization first.


In [ ]:

text = "The infant learned the new word."
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text)

print("Original text:")
print(text)
print("\nTokens:")
print(tokens)
#print("\nToken IDs:")
#print(token_ids)


**Notice** two things:
- The tokenizer adds an odd "Ġ" symbol. This means that there is a space before the word (so you don't need the symbol for "The" because that's the first word, but it's added to "infant" because there is a space between "The" and "infant".
- Punctuation gets its own token (e.g., ".")




### 1.1 Try it yourself

Replace the sentence below with your own sentence and run the cell.
Try:
- a simple sentence
- a sentence with punctuation
- a hyphenated word like `well-known`
- a name like `ChatGPT`
- a made-up word like `supercalifragilisticexpialidocious`


In [ ]:

my_text = "The student inspected the code supercalifragilisticexpialidocious."
print(tokenizer.tokenize(my_text))



## 2. Looking at next-token predictions

Now let's ask the model:

> Given a prompt, what tokens does it think are most likely to come next?

This is the heart of language modeling: it's the goal the model is trained on and what it fundamentally is trying to do (which is why these models are sometimes called "fancy autocomplete")


In [ ]:

# This cell creates a function for showing the top predicted tokens. Make sure to run this first!
def show_top_next_tokens(prompt, top_k=10):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)

    next_token_logits = outputs.logits[0, -1]
    probs = torch.softmax(next_token_logits, dim=-1)
    top_probs, top_ids = torch.topk(probs, top_k)

    rows = []
    for rank, (tok_id, prob) in enumerate(zip(top_ids.tolist(), top_probs.tolist()), start=1):
        token_text = tokenizer.decode([tok_id])
        rows.append({
            "rank": rank,
            "token": repr(token_text),
            "probability": round(prob, 4)
        })
    return pd.DataFrame(rows)


In [ ]:
# Now run the prompt below to see the top-ten next predicted words
prompt = "The capital of the UK is"
show_top_next_tokens(prompt, top_k=10)


### Visualizing the prediction distribution

The next cell shows the distribution of the model's top next-token predictions.

This helps you see:
- which token is most likely
- whether the model is very certain or spread out across many choices

*Note: Notice that the most likely token is NOT London. Can you think of some reasons why?*


In [ ]:

def plot_top_next_tokens(prompt, top_k=10):
    df = show_top_next_tokens(prompt, top_k=top_k).copy()
    df["token_clean"] = df["token"].str.replace("^'|'$", "", regex=True)

    plt.figure(figsize=(10, 5))
    plt.bar(df["token_clean"], df["probability"])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Probability")
    plt.xlabel("Next token")
    plt.title(f"Top next-token predictions for: {prompt!r}")
    plt.tight_layout()
    plt.show()

plot_top_next_tokens("The capital of the UK is", top_k=10)



### 2.1. Try it yourself

**Use the code chunk below to do each task.**

2.1.1 In the code chunk below, try a few different prompts and rerun. *List the prompts you tried below*

- [prompt 1]
- [prompt 2]
- ...

2.1.2 Expand the code below to show the top 20 (top 50) entries. *Hint: use the top_k parameter in the function call*


### 2.2. Questions

**2.2.1. Can you describe what the probability distribution looks like?**
*(Write 1–2 sentences in a comment before moving on)*

[replace this placeholder with your answer!]

**2.2.2. Why do you think the model might assign small probabilities to many words?**
*(Write 1–2 sentences in a comment before moving on)*

[replace this placeholder with your answer!]





In [ ]:

# testing code chunk: try out your different prompts and explore the probability distributions by editing the prompts and functions below
your_prompt = "Peanut butter and jelly is a tasty"
plot_top_next_tokens(your_prompt, top_k=10)



## 3. Generating text step by step

When GPT writes a sentence, it does not produce the whole answer at once.

Instead, it does this loop:

1. Read the tokens so far
2. Predict a distribution over the next token
3. Choose one token
4. Add it to the context
5. Repeat

Let's generate a short continuation. Try re-running the code chunk a few times - the story will change each time!


In [ ]:

prompt = "Once upon a time, a robot learned to"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    generated = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=True,
        temperature=0.8,
        top_p=0.95
    )

print(tokenizer.decode(generated[0], skip_special_tokens=True))


### Bonus question 1 (optional)

Looking at the documentation for the ```generate()``` function [here](https://huggingface.co/docs/transformers/main_classes/text_generation) will help you understand which piece of code does what and how each parameter informs next word generation.

What parameter in the code above can you change so that the continuation becomes more consistent? (What parameter(s) control(s) how "noisy" the choice of next word is?)

[type your answer here]



### Visualizing generation step by step

The next cell slows generation down so you can watch the sentence grow one token at a time.
This makes the autoregressive loop more concrete.


In [ ]:

def generate_step_by_step(prompt, steps=8):
    current_text = prompt
    print("Starting text:")
    print(repr(current_text))
    print("-" * 50)

    for step in range(steps):
        inputs = tokenizer(current_text, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)

        next_token_logits = outputs.logits[0, -1]
        probs = torch.softmax(next_token_logits, dim=-1)
        next_id = torch.argmax(probs).item()
        next_token = tokenizer.decode([next_id])

        current_text += next_token

        print(f"Step {step+1}: added token {repr(next_token)}")
        print(current_text)
        print("-" * 50)

    return current_text

generate_step_by_step("The tiny dragon wanted to", steps=8)


### 3.1. Try it yourself

In the code chunk below, try out your own continuation. You can vary the prompt and the number of continuation steps (i.e., new words you add to the story).

In [ ]:
#add your own prompt below and try varying the step size
generate_step_by_step("Imagine that", steps=8)


## 4. Self-attention: how a token looks back at earlier tokens

Here is the main intuition behind self-attention:

When GPT is processing a token, it can look back at earlier tokens and decide:

> Which previous tokens are most useful right now (for my main job, predicting the next word)?

For example, in the sentence:

> *The animal didn't cross the street because **it** was tired.*

the token `"it"` may attend strongly to `"animal"`, because `"it"` refers to `"animal"` (the subject that showed up earlier in the sentence).

Self-attention gives the model a flexible way to connect words across a sentence.

We will inspect attention from one layer and one attention head.

Note: To get a firmer understanding of how attention works (without math) I recommend [this video](https://www.youtube.com/watch?v=cGMnNi4XGvc).



**Helpful constraint**

Check max values:

```python
print(model.config.n_layer, model.config.n_head)
```

These are the maximum ranges for layers (6 total) and heads (12 total)
- layers: 0 to 5  
- heads: 0 to 11

---

**How to read attention (simpler strategy)**

Focus on ONE token:
1. Find its position
2. Look across its row
3. Identify the strongest weights

That shows what the model is "looking at".


In [ ]:
def get_model_outputs(text):
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    return inputs, outputs

text = "The animal didn't cross the street because it was tired."
inputs, outputs = get_model_outputs(text)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(tokens)


In [ ]:

def plot_attention(text, layer=0, head=0):
    inputs, outputs = get_model_outputs(text)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    # attention shape: [batch, heads, query_positions, key_positions]
    attn = outputs.attentions[layer][0, head].detach().cpu()

    plt.figure(figsize=(10, 8))
    plt.imshow(attn, aspect="auto")
    plt.colorbar()
    plt.xticks(range(len(tokens)), tokens, rotation=90)
    plt.yticks(range(len(tokens)), tokens)
    plt.xlabel("Tokens being attended to")
    plt.ylabel("Current token")
    plt.title(f"Attention map | layer={layer}, head={head}")
    plt.show()

plot_attention(text, layer=4, head=0)



### How to read the attention map

- Each **row** is the current token doing the looking
- Each **column** is a previous token it might look at
- Brighter values mean **more attention**

### Important caution
GPT contains many, many attention heads. Different heads and layers all do different things, and it can often be quite difficult to understand what any given attention head is doing in isolation.

Still, this gives you a useful window into what's happening "under the hood" in the model.


### 4.1 Question

**`it` and `was` have a strong attention weight on animal. Why do you think that might be, and how would you describe what this attention head's "job" is in the model?**

*(Write 1–2 sentences in a comment before moving on)*

[replace this placeholder with your answer!]



### 4.2 Try it yourself

4.2.1. Change the sentence in the code chunk below

Good test sentences:
- `"The trophy doesn't fit in the suitcase because it is too large."`
- `"Maria thanked Susan because she helped with the project."`
- `"The book on the table near the lamp was old."`

Look for tokens like `it`, `she`, or `was`. Which earlier tokens do they attend to?

4.2.2. Try changing the layer and the head. Observe how attention is changing as you change the layer/head. *Note: sometimes it'll sort of make sense, sometimes it might be very hard to understand why attention looks the way it does.*

4.2.3. Once you find a layer/head that has an interesting attention pattern, describe it below (1-2 sentences).

Your sentence:

Layer:

Head:

[your description of the pattern/ what you found interesting]



In [ ]:
#edit this part of the code for 4.2
your_sentence = "The trophy doesn't fit in the suitcase because it is too large."
plot_attention(your_sentence, layer=4, head=0)

## 5. Static word similarity: words as points in space

Before we think about how a word changes across sentences, let's start with a simpler idea:

> A model represents tokens as vectors.

A **vector** is just a list of numbers. GPT-style models begin with an **input embedding** for each token in the vocabulary. These input embeddings are "static" in the sense that the token starts with the same vector each time it appears.

So before the model reads a full sentence, the token for `" dog"` has one starting vector, the token for `" cat"` has another starting vector, and so on.

If the model has learned useful structure, we might expect semantically similar words to be closer together:
- `"cat"` and `"dog"` might be relatively close
- `"car"` and `"truck"` might be relatively close
- `"cat"` and `"truck"` might be farther apart

### One tokenization caution

For GPT-2 style tokenizers, `"cat"` and `" cat"` can be different tokens. In normal sentences, most words appear with a space before them, so below we will usually look up words as `" " + word`.

Run the next cell to define a few helper functions.


In [ ]:
import torch.nn.functional as F

def encode_as_single_common_token(word):
    """Encode a word as it usually appears inside a sentence: with a leading space."""
    token_ids = tokenizer.encode(" " + word, add_special_tokens=False)

    if len(token_ids) != 1:
        print(f"Warning: {word!r} was split into multiple tokens:", tokenizer.convert_ids_to_tokens(token_ids))
        print("For this simple demo, using the first token only.")

    return token_ids[0]

def get_static_embedding(word):
    token_id = encode_as_single_common_token(word)
    return model.get_input_embeddings().weight[token_id].detach().cpu()

def cosine(a, b):
    return F.cosine_similarity(a, b, dim=0).item()

def static_similarity(word_a, word_b):
    return cosine(get_static_embedding(word_a), get_static_embedding(word_b))


### 5.1 Compare a few word pairs

The next cell compares **cosine similarity** for several pairs of words.

Cosine similarity is a rough measure of whether two vectors point in similar directions:
- closer to **1** means more similar
- closer to **0** means less similar
- negative values mean the vectors point in different directions

Do not worry too much about the exact numbers. Focus on the pattern.


In [ ]:
pairs = [
    ("cat", "dog"),
    ("puppy", "dog"),
    ("car", "truck"),
    ("bus", "truck"),
    ("cat", "truck"),
    ("puppy", "table"),
    ("car", "banana")
]

rows = []
for w1, w2 in pairs:
    rows.append({
        "word 1": w1,
        "word 2": w2,
        "cosine similarity": round(static_similarity(w1, w2), 3)
    })

static_similarity_df = pd.DataFrame(rows)
static_similarity_df


### 5.2 Visualize static embeddings

Now let's plot several static word embeddings in 2D.

The real embeddings have many dimensions, so we will use a simple projection method called PCA to squash them into two dimensions for visualization.

This plot is only a rough map. Do not over-interpret exact distances. The main question is whether related words tend to land near each other. (Pretty cool that they do, right?!?!)


In [ ]:
def pca_2d_torch(matrix):
    """Simple PCA to 2D using torch, so we do not need any extra packages."""
    X = matrix - matrix.mean(dim=0, keepdim=True)
    U, S, Vh = torch.linalg.svd(X, full_matrices=False)
    return X @ Vh[:2].T

word_groups = {
    "animals": ["cat", "dog", "puppy", "kitten", "horse", "mouse"],
    "vehicles": ["car", "truck", "bus", "train", "taxi", "bicycle"],
    "foods": ["apple", "banana", "bread", "pizza", "cheese", "cookie"]
}

words = [w for group in word_groups.values() for w in group]
embeddings = torch.stack([get_static_embedding(w) for w in words])
coords = pca_2d_torch(embeddings)

plt.figure(figsize=(9, 6))
plt.scatter(coords[:, 0], coords[:, 1])

for i, word in enumerate(words):
    plt.annotate(word, (coords[i, 0], coords[i, 1]))

plt.xlabel("PCA dimension 1")
plt.ylabel("PCA dimension 2")
plt.title("Static input embeddings projected into 2D")
plt.tight_layout()
plt.show()


### 5.3 Question

**5.3.1. Do words from the same category tend to appear closer together in the plot? Give one example from the visualization.**

*(Write 1–2 sentences below.)*

[replace this placeholder with your answer!]

**5.3.2. Are there any relationships that surprised you? Give one example from the visualization.**

*(Write 1–2 sentences below.)*

[replace this placeholder with your answer!]


## 5.4. Bonus question (optional)

Write 1–2 lines of code to compare two new static word pairs.

For example:
- `"whale"` vs. `"dolphin"`
- `"whale"` vs. `"table"`

Use the `static_similarity()` helper from Section 5.


In [ ]:
# Bonus question code space
# Example format:
# static_similarity("whale", "dolphin")

## 6. Context-sensitive representations: the same word can move

Static embeddings are only the starting point for how the model represents words (really: tokens).

Once a token appears in a sentence, the transformer updates its representation across layers. That means the model's representation of a word can change depending on the surrounding context.

This is especially important for ambiguous words.

For example, `"bat"` can mean:
- an animal
- sports equipment

Next, we will see how the representation of bat changes within the model, depending on its surrounding context. To see how the representation changes, we will ask how similar `"bat"` is to different words (some animal-related, some sports-related), depending on its context.

> Does `"bat"` become relatively more similar to animal-related words in the animal sentence, and relatively more similar to sports-related words in the baseball sentence?

This gives us what we'll call a **similarity profile** for each instance of `"bat"`.

*Note: There's a lot of code here, don't worry too much about it. Focus on the plots below and understanding the main idea behind them.*


In [ ]:
animal_sentence = "At dusk, the bat flew out of the cave and hunted insects."
sports_sentence = "During the game, the player swung the bat and hit the baseball."

probe_words = [
     "insect", "wing","vampire",
      "player", "game","glove"
]

def get_contextual_token_vector(sentence, target_word, layer=None):
    """Return the hidden-state vector for the first token matching target_word."""
    if layer is None:
        layer = model.config.n_layer  # final hidden-state layer

    inputs = tokenizer(sentence, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    # Look for target inside the cleaned token text.
    for i, tok in enumerate(tokens):
        cleaned_tok = tok.replace("Ġ", "").lower()
        if target_word.lower() in cleaned_tok:
            return outputs.hidden_states[layer][0, i].detach().cpu(), tokens, i

    raise ValueError(f"Could not find {target_word!r}. Tokens were: {tokens}")

def centered_cosine(a, b, center):
    """Cosine similarity after subtracting a shared center vector.

    This often makes relative differences easier to see in transformer hidden states.
    """
    return cosine(a - center, b - center)

def make_probe_vectors(probe_words, layer=None):
    """Make contextual vectors for probe words in short neutral sentences."""
    probe_vecs = {}
    probe_tokens = {}

    for word in probe_words:
        sentence = f"This is about {word}."
        vec, toks, pos = get_contextual_token_vector(sentence, word, layer=layer)
        probe_vecs[word] = vec
        probe_tokens[word] = toks

    return probe_vecs, probe_tokens

layer_to_inspect = model.config.n_layer  # final layer

animal_bat_vec, animal_tokens, animal_pos = get_contextual_token_vector(animal_sentence, "bat", layer=layer_to_inspect)
sports_bat_vec, sports_tokens, sports_pos = get_contextual_token_vector(sports_sentence, "bat", layer=layer_to_inspect)

probe_vecs, probe_tokens = make_probe_vectors(probe_words, layer=layer_to_inspect)

# Center using all the vectors we are comparing.
all_vecs = torch.stack([animal_bat_vec, sports_bat_vec] + list(probe_vecs.values()))
center = all_vecs.mean(dim=0)

rows = []
for word in probe_words:
    rows.append({
        "probe word": word,
        "similarity to bat in animal sentence": centered_cosine(animal_bat_vec, probe_vecs[word], center),
        "similarity to bat in sports sentence": centered_cosine(sports_bat_vec, probe_vecs[word], center)
    })

context_shift_df = pd.DataFrame(rows)
context_shift_df["shift toward sports context"] = (
    context_shift_df["similarity to bat in sports sentence"] -
    context_shift_df["similarity to bat in animal sentence"]
)

#print("Animal sentence tokens:")
#for i, tok in enumerate(animal_tokens):
#    marker = "<-- target" if i == animal_pos else ""
#    print(i, tok, marker)

#print("\nSports sentence tokens:")
#for i, tok in enumerate(sports_tokens):
#    marker = "<-- target" if i == sports_pos else ""
#    print(i, tok, marker)

context_shift_df.round(3)


### 6.1 Visualize the similarity profile

The table is useful, but the plot is easier to read.

If context matters, we might see:
- the animal-sentence `"bat"` closer to words like `"insect"`, `"wing"`, or `"vampire"`
- the sports-sentence `"bat"` closer to words like `"player"`, `"game"`, or `"glove"`

The pattern will not be perfect, because this is a tiny model and a very simple analysis. The important thing is the **relative shift**.


In [ ]:
plot_df = context_shift_df.set_index("probe word")[
    ["similarity to bat in animal sentence", "similarity to bat in sports sentence"]
]

ax = plot_df.plot(kind="bar", figsize=(11, 5))
plt.ylabel("Centered cosine similarity")
plt.xlabel("Probe word")
plt.title("How the representation of 'bat' shifts with context")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### 6.2 Another way to see the shift

The next plot subtracts the animal-context similarity from the sports-context similarity.

Positive values mean the probe word is **more similar to `"bat"` in the sports sentence**.

Negative values mean the probe word is **more similar to `"bat"` in the animal sentence**.


In [ ]:
shift_df = context_shift_df.sort_values("shift toward sports context")

plt.figure(figsize=(10, 5))
plt.bar(shift_df["probe word"], shift_df["shift toward sports context"])
plt.axhline(0, linewidth=1)
plt.ylabel("sports-context similarity minus animal-context similarity")
plt.xlabel("Probe word")
plt.title("Relative shift in the representation of 'bat'")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### 6.3 Question

**Which probe words shifted most toward the sports-context `"bat"`? Which shifted most toward the animal-context `"bat"`?**

*(Write 1–2 sentences below.)*

[replace this placeholder with your answer!]


### 6.4 Try it yourself

Try replacing `"bat"` with another ambiguous word.

A couple of ideas:
- `"seal"`: animal vs. official stamp
- `"crane"`: bird vs. construction machine
- `"mouse"`: animal vs. computer device

You will need to change:
1. the two sentences
2. the `target_word`
3. the probe words `my_probe_words`

**Your target word:**

**Your two sentences:**

**Your probe words:**

**What changed in the similarity profile?**

[replace this placeholder with your answer!]


In [ ]:
# I've filled out one example, but try your own example here!

sentence_a = "The seal swam through the cold ocean water."
sentence_b = "The official pressed the seal into the warm wax."

target_word = "seal"

my_probe_words = [
    "fin", "water", "swim",
    "official", "letter", "document"
]

vec_a, toks_a, pos_a = get_contextual_token_vector(sentence_a, target_word, layer=layer_to_inspect)
vec_b, toks_b, pos_b = get_contextual_token_vector(sentence_b, target_word, layer=layer_to_inspect)
my_probe_vecs, _ = make_probe_vectors(my_probe_words, layer=layer_to_inspect)

all_vecs = torch.stack([vec_a, vec_b] + list(my_probe_vecs.values()))
center = all_vecs.mean(dim=0)

rows = []
for word in my_probe_words:
    rows.append({
        "probe word": word,
        "similarity in sentence A": centered_cosine(vec_a, my_probe_vecs[word], center),
        "similarity in sentence B": centered_cosine(vec_b, my_probe_vecs[word], center),
    })

my_shift_df = pd.DataFrame(rows)
my_shift_df["sentence B minus sentence A"] = (
    my_shift_df["similarity in sentence B"] -
    my_shift_df["similarity in sentence A"]
)

display(my_shift_df.round(3))

my_shift_df.set_index("probe word")[["similarity in sentence A", "similarity in sentence B"]].plot(
    kind="bar",
    figsize=(11, 5)
)
plt.ylabel("Centered cosine similarity")
plt.title(f"How the representation of {target_word!r} shifts with context")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 7. Final Questions

Try answering these in your own words.

### Question 7.1
What is the difference between a **word** and a **token**?

### Question 7.2
Why is it essential for language processing that the same word has different internal representations in different sentences?

### Question 7.3
Why might a model that only predicts the next token still learn a lot about meaning?

You can write your answers in a new markdown cell below.

### Question 7.1

[your answer]

### Question 7.2

[your answer]

### Question 7.3

[your answer]


## 8. Summing up

In this notebook, you got a better sense of the inner workings of GPT. A GPT model works by repeatedly doing something surprisingly simple:

> turn text into tokens, build context-sensitive representations, attend to what matters, and predict the next token.

You visualized:
- **prediction** as a probability distribution over next tokens
- **generation** as a sentence growing one token at a time
- **self-attention** as maps and token-specific attention weights
- **representations** as vectors projected into 2D
- **context-sensitivity** of word representations
